# Surface-code Knill fixed/adaptive sweeps

This notebook is an experiment driver for the fixed-round and two-level adaptive Knill APIs. It intentionally defaults to a tiny smoke configuration. Do not infer thresholds or production LER from these defaults.

The adaptive control is Bell-pair synchronized: each teleportation decodes the `|0_L>` and `|+_L>` short patches independently, then uses the OR of their patch-level extension decisions. Low-confidence patches continue the same physical `FlipSimulator` shot; no independent long circuit is sampled. The code-capacity BP-LSD prior is currently uniform per decoder variable, not a circuit-derived effective prior. Both static and adaptive backends consume the derived seeds; Stim seeds are reproducible only for compatible Stim versions and machines.

In [ ]:
from __future__ import annotations

import hashlib
import json
import platform
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymatching
import ldpc
import stim

import hex_qec
from hex_qec.circuit_generation import get_parity_check_matrices
from hex_qec.decoders import make_bplsd_decoder_generator
from hex_qec.modularisation import AdaptiveSERounds, generate_adaptive_state_prep_module
from hex_qec.protocols import knill_online_offline, knill_online_offline_adaptive
from hex_qec.simulation import (
    AlwaysLongPolicy,
    AlwaysShortPolicy,
    ClusterLLRPolicy,
    StatefulAdaptiveStatePrepExecutor,
)

SMOKE_TEST = True
PHYSICAL_ERROR_RATES = [0.001] if SMOKE_TEST else [1e-4, 3e-4, 1e-3, 3e-3]
DISTANCES = [3] if SMOKE_TEST else [3, 5, 7]
FIXED_ROUNDS = [1, 2] if SMOKE_TEST else [1, 2, 3, 5]
DECODERS = ["mwpm", "bplsd"]
NUM_TELEPORTATIONS = 1
PAULI = "z"
SHORT_ROUNDS = [1] if SMOKE_TEST else [1, 2, 3]
LONG_ROUNDS_RULE = "distance"  # also accepts an integer or callable
THRESHOLDS = [0.01] if SMOKE_TEST else [0.001, 0.003, 0.01, 0.03]
MAX_SHOTS = 4 if SMOKE_TEST else 10_000
MAX_ERRORS_BEFORE_HALTING = 10
BASE_SEED = 1729
CLUSTER_LLR_ALPHA = 2.0
SURFACE_CODE = True
BPLSD_OPTIONS = dict(
    max_iter=30,
    bp_method="minimum_sum",
    lsd_method="LSD_0",
    lsd_order=0,
    always_run_lsd=True,
)

def long_rounds_for(distance):
    if callable(LONG_ROUNDS_RULE):
        return int(LONG_ROUNDS_RULE(distance))
    if LONG_ROUNDS_RULE == "distance":
        return int(distance)
    return int(LONG_ROUNDS_RULE)

REPO_ROOT = Path(hex_qec.__file__).resolve().parents[2]
RESULTS_ROOT = REPO_ROOT / "results"
FIXED_CSV = RESULTS_ROOT / "fixed_knill_surface.csv"
ADAPTIVE_CSV = RESULTS_ROOT / "adaptive_knill_surface.csv"
ADAPTIVE_SHOTS = RESULTS_ROOT / "adaptive_shots"
FIGURES_ROOT = RESULTS_ROOT / "figures"

def git_sha():
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True).strip()
    except (OSError, subprocess.CalledProcessError):
        return "unknown"

RUN_METADATA = {
    "git_sha": git_sha(),
    "python": sys.version,
    "platform": platform.platform(),
    "stim": getattr(stim, "__version__", "unknown"),
    "ldpc": getattr(ldpc, "__version__", "unknown"),
    "pymatching": getattr(pymatching, "__version__", "unknown"),
    "hex_source": str(REPO_ROOT / "src" / "hex_qec"),
    "base_seed": BASE_SEED,
    "noise": "Hex simplified circuit-level Pauli model using one physical_error probability",
    "code_capacity_prior": "uniform per data/error variable for BP-LSD; not circuit-derived",
    "bplsd_options": BPLSD_OPTIONS,
    "cluster_llr_alpha": CLUSTER_LLR_ALPHA,
    "adaptive_pair_control": "synchronized OR of |0_L> and |+_L> patch extension decisions",
    "fixed_seed_behavior": "derived seed passed to Stim compiled sampler",
}
print(json.dumps(RUN_METADATA, indent=2, default=str))

In [ ]:
def wilson_interval(errors, shots, z=1.959963984540054):
    if shots <= 0:
        return np.nan, np.nan
    p = errors / shots
    denominator = 1 + z**2 / shots
    center = (p + z**2 / (2 * shots)) / denominator
    radius = z * np.sqrt(p * (1 - p) / shots + z**2 / (4 * shots**2)) / denominator
    return max(0.0, center - radius), min(1.0, center + radius)

def stable_seed(*parts):
    payload = json.dumps(parts, sort_keys=True, default=str).encode()
    return (BASE_SEED + int.from_bytes(hashlib.sha256(payload).digest()[:8], "little")) % (2**63 - 1)

def ensure_result_dirs():
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
    ADAPTIVE_SHOTS.mkdir(parents=True, exist_ok=True)
    FIGURES_ROOT.mkdir(parents=True, exist_ok=True)

def load_checkpoint(path):
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

def checkpoint_key(row, key_columns):
    return tuple(row[column] for column in key_columns)

def checkpoint_has(path, row, key_columns):
    old = load_checkpoint(path)
    if old.empty:
        return False
    key = checkpoint_key(row, key_columns)
    return any(checkpoint_key(item, key_columns) == key for item in old.to_dict("records"))

def checkpoint_append(path, row, key_columns, overwrite=False):
    old = load_checkpoint(path)
    key = checkpoint_key(row, key_columns)
    if not overwrite and not old.empty:
        existing = {tuple(item[column] for column in key_columns) for item in old.to_dict("records")}
        if key in existing:
            return old, False
    if overwrite and not old.empty:
        mask = np.ones(len(old), dtype=bool)
        for column in key_columns:
            mask &= old[column].to_numpy() == row[column]
        old = old.loc[~mask]
    updated = pd.concat([old, pd.DataFrame([row])], ignore_index=True)
    updated.to_csv(path, index=False)
    return updated, True

## Decoder factories and confidence aggregation

The BP-LSD adapter uses a uniform code-capacity channel when Hex calls the generator without `weights`. In the non-matchable DEM path, supplied DEM probabilities are passed as decoder-variable probabilities. The conservative `max_all_components` aggregator is an explicit experiment choice over the four CSS decoder results.

In [ ]:
def decoder_generators(name, physical_error):
    if name == "mwpm":
        generator = pymatching.Matching.from_check_matrix
        return generator, generator, True
    if name == "bplsd":
        generator = make_bplsd_decoder_generator(
            physical_error, alpha=CLUSTER_LLR_ALPHA, **BPLSD_OPTIONS
        )
        return generator, generator, False
    raise ValueError(f"unknown decoder {name!r}")

def max_all_components(results):
    values = [result.confidence for result in results if result.confidence is not None]
    return np.max(np.stack(values), axis=0) if values else None

def max_dem_only(results):
    values = [result.confidence for result in results[:2] if result.confidence is not None]
    return np.max(np.stack(values), axis=0) if values else None

CONFIDENCE_AGGREGATORS = {
    "max_all_components": max_all_components,
    "max_dem_only": max_dem_only,
}

In [ ]:
def run_fixed_point(physical_error, distance, rounds, decoder_name, *, max_shots=MAX_SHOTS):
    parity_checks = get_parity_check_matrices("surface", distance)
    online, offline, matchable = decoder_generators(decoder_name, physical_error)
    seed = stable_seed("fixed", physical_error, distance, rounds, decoder_name, NUM_TELEPORTATIONS, PAULI)
    start = time.perf_counter()
    shots, errors = knill_online_offline(
        parity_checks, rounds, online, offline, matchable, physical_error,
        max_shots, MAX_ERRORS_BEFORE_HALTING, PAULI, NUM_TELEPORTATIONS,
        seed=seed,
        surface_code=SURFACE_CODE,
    )
    low, high = wilson_interval(errors, shots)
    return {
        "physical_error": physical_error, "distance": distance, "rounds": rounds,
        "decoder": decoder_name, "num_teleportations": NUM_TELEPORTATIONS,
        "pauli": PAULI, "shots": shots, "logical_errors": errors,
        "logical_error_rate": errors / shots if shots else 0.0,
        "ler_ci_low": low, "ler_ci_high": high,
        "runtime_seconds": time.perf_counter() - start, "seed": seed,
        "surface_code": SURFACE_CODE,
    }

def run_fixed_sweep(overwrite=False):
    ensure_result_dirs()
    table = load_checkpoint(FIXED_CSV)
    key = ["physical_error", "distance", "rounds", "decoder", "num_teleportations", "pauli", "surface_code"]
    for p in PHYSICAL_ERROR_RATES:
        for distance in DISTANCES:
            for rounds in FIXED_ROUNDS:
                for decoder_name in DECODERS:
                    identity = {"physical_error": p, "distance": distance, "rounds": rounds,
                                "decoder": decoder_name, "num_teleportations": NUM_TELEPORTATIONS,
                                "pauli": PAULI, "surface_code": SURFACE_CODE}
                    if checkpoint_has(FIXED_CSV, identity, key) and not overwrite:
                        continue
                    row = run_fixed_point(p, distance, rounds, decoder_name)
                    table, added = checkpoint_append(FIXED_CSV, row, key, overwrite=overwrite)
                    if added:
                        print("fixed", row)
    return table


In [ ]:
def run_adaptive_point(physical_error, distance, short_rounds, threshold, aggregator_name, *, max_shots=MAX_SHOTS):
    parity_checks = get_parity_check_matrices("surface", distance)
    online, offline, matchable = decoder_generators("bplsd", physical_error)
    long_rounds = long_rounds_for(distance)
    seed = stable_seed("adaptive", physical_error, distance, short_rounds, threshold, aggregator_name, NUM_TELEPORTATIONS, PAULI)
    schedule = AdaptiveSERounds(short_rounds, long_rounds, ClusterLLRPolicy(threshold))
    start = time.perf_counter()
    result = knill_online_offline_adaptive(
        parity_checks, schedule, online, offline, matchable, physical_error,
        max_shots, MAX_ERRORS_BEFORE_HALTING, PAULI, NUM_TELEPORTATIONS,
        confidence_aggregator=CONFIDENCE_AGGREGATORS[aggregator_name],
        detail_level="analysis", batch_size=min(256, max_shots), seed=seed,
        surface_code=SURFACE_CODE,
    )
    pairs = result.bell_pair_stats
    if not pairs:
        raise RuntimeError("adaptive result did not contain Bell-pair statistics")
    pair_long = sum(item.long_count or 0 for item in pairs)
    pair_short = sum(item.short_count or 0 for item in pairs)
    pair_total = pair_long + pair_short
    row = {
        "physical_error": physical_error, "distance": distance,
        "short_rounds": short_rounds, "long_rounds": long_rounds,
        "threshold": threshold, "cluster_llr_alpha": CLUSTER_LLR_ALPHA,
        "confidence_aggregator": aggregator_name,
        "num_teleportations": NUM_TELEPORTATIONS, "pauli": PAULI,
        "shots": result.shots, "logical_errors": result.logical_errors,
        "logical_error_rate": result.logical_error_rate,
        "ler_ci_low": wilson_interval(result.logical_errors, result.shots)[0],
        "ler_ci_high": wilson_interval(result.logical_errors, result.shots)[1],
        "runtime_seconds": result.summary.runtime_seconds, "seed": seed,
        "surface_code": SURFACE_CODE,
        "pair_fallback_rate": pair_long / pair_total if pair_total else 0.0,
        "pair_short_fraction": pair_short / pair_total if pair_total else 0.0,
        "pair_long_fraction": pair_long / pair_total if pair_total else 0.0,
        "mean_pair_risk": float(np.nanmean(result.per_shot["pair_risk"]))
            if result.per_shot is not None and np.any(np.isfinite(result.per_shot["pair_risk"])) else np.nan,
        "mean_effective_rounds": float(np.mean([item.mean_effective_rounds for item in pairs])),
        "z_only_fallback_fraction": float(np.mean([item.z_only_fallback_fraction for item in pairs])),
        "x_only_fallback_fraction": float(np.mean([item.x_only_fallback_fraction for item in pairs])),
        "both_fallback_fraction": float(np.mean([item.both_fallback_fraction for item in pairs])),
        "mean_z_patch_risk": float(np.nanmean(result.per_shot["confidence"][:, 0])),
        "mean_x_patch_risk": float(np.nanmean(result.per_shot["confidence"][:, 1])),
    }
    shot_key = hashlib.sha256(json.dumps(row, sort_keys=True, default=str).encode()).hexdigest()[:16]
    if result.per_shot is not None:
        ensure_result_dirs()
        np.savez_compressed(ADAPTIVE_SHOTS / f"{shot_key}.npz", **result.per_shot)
    row["runtime_seconds"] = time.perf_counter() - start
    return row

def run_adaptive_sweep(overwrite=False):
    ensure_result_dirs()
    table = load_checkpoint(ADAPTIVE_CSV)
    key = ["physical_error", "distance", "short_rounds", "threshold", "confidence_aggregator", "num_teleportations", "pauli", "surface_code"]
    for p in PHYSICAL_ERROR_RATES:
        for distance in DISTANCES:
            for short_rounds in SHORT_ROUNDS:
                if distance < short_rounds:
                    continue
                for threshold in THRESHOLDS:
                    for aggregator_name in CONFIDENCE_AGGREGATORS:
                        identity = {"physical_error": p, "distance": distance,
                                    "short_rounds": short_rounds, "threshold": threshold,
                                    "confidence_aggregator": aggregator_name,
                                    "num_teleportations": NUM_TELEPORTATIONS, "pauli": PAULI,
                                    "surface_code": SURFACE_CODE}
                        if checkpoint_has(ADAPTIVE_CSV, identity, key) and not overwrite:
                            continue
                        row = run_adaptive_point(p, distance, short_rounds, threshold, aggregator_name)
                        table, added = checkpoint_append(ADAPTIVE_CSV, row, key, overwrite=overwrite)
                        if added:
                            print("adaptive", row)
    return table

## Plotting helpers

These functions create figures on demand. The adaptive primary coordinate is measured `mean_effective_rounds`, not configured `short_rounds`.

In [ ]:
def plot_fixed_ler_vs_error(table, distance, rounds=None):
    view = table[table.distance == distance]
    if rounds is not None:
        view = view[view.rounds.isin(rounds)]
    fig, ax = plt.subplots()
    for (decoder, round_count), group in view.groupby(["decoder", "rounds"]):
        group = group.sort_values("physical_error")
        ax.errorbar(group.physical_error, group.logical_error_rate,
                    yerr=[group.logical_error_rate - group.ler_ci_low, group.ler_ci_high - group.logical_error_rate],
                    marker="o", label=f"{decoder}, rounds={round_count}")
    ax.set_yscale("log"); ax.set_xlabel("physical error"); ax.set_ylabel("LER"); ax.legend()
    return fig, ax

def plot_fixed_ler_by_distance(table, rounds=None, decoder=None):
    view = table if decoder is None else table[table.decoder == decoder]
    if rounds is not None: view = view[view.rounds == rounds]
    fig, ax = plt.subplots()
    for distance, group in view.groupby("distance"):
        group = group.sort_values("physical_error")
        ax.plot(group.physical_error, group.logical_error_rate, marker="o", label=f"d={distance}")
    ax.set_yscale("log"); ax.set_xlabel("physical error"); ax.set_ylabel("LER"); ax.legend()
    return fig, ax

def plot_fixed_decoder_comparison(table, distance, rounds, physical_error):
    view = table[(table.distance == distance) & (table.rounds == rounds) & (table.physical_error == physical_error)]
    fig, ax = plt.subplots()
    ax.errorbar(view.decoder, view.logical_error_rate,
                yerr=[view.logical_error_rate - view.ler_ci_low, view.ler_ci_high - view.logical_error_rate], marker="o")
    ax.set_yscale("log"); ax.set_ylabel("LER")
    return fig, ax

def plot_adaptive_effective_rounds(table, distance=None, physical_error=None):
    view = table
    if distance is not None: view = view[view.distance == distance]
    if physical_error is not None: view = view[view.physical_error == physical_error]
    fig, ax = plt.subplots()
    for aggregator, group in view.groupby("confidence_aggregator"):
        group = group.sort_values("mean_effective_rounds")
        ax.plot(group.mean_effective_rounds, group.logical_error_rate, marker="o", label=aggregator)
    ax.set_yscale("log"); ax.set_xlabel("mean effective SE rounds"); ax.set_ylabel("LER"); ax.legend()
    return fig, ax

def plot_adaptive_ler_vs_short_rounds(table, distance=None, threshold=None):
    view = table
    if distance is not None: view = view[view.distance == distance]
    if threshold is not None: view = view[view.threshold == threshold]
    fig, ax = plt.subplots()
    for aggregator, group in view.groupby("confidence_aggregator"):
        group = group.sort_values("short_rounds")
        ax.errorbar(group.short_rounds, group.logical_error_rate,
                    yerr=[group.logical_error_rate - group.ler_ci_low,
                          group.ler_ci_high - group.logical_error_rate],
                    marker="o", label=aggregator)
    ax.set_yscale("log"); ax.set_xlabel("short SE rounds"); ax.set_ylabel("LER"); ax.legend()
    return fig, ax

def plot_adaptive_thresholds(table, distance, physical_error, short_rounds):
    view = table[(table.distance == distance) & (table.physical_error == physical_error) &
                 (table.short_rounds == short_rounds)]
    fig, ax = plt.subplots()
    for aggregator, group in view.groupby("confidence_aggregator"):
        group = group.sort_values("threshold")
        ax.plot(group.threshold, group.mean_effective_rounds, marker="o", label=aggregator)
    ax.set_xlabel("Cluster LLR threshold"); ax.set_ylabel("mean effective SE rounds"); ax.legend()
    return fig, ax

def plot_confidence_histogram(per_shot, event_index=0, bins=20):
    values = np.asarray(per_shot["confidence"])[:, event_index]
    values = values[np.isfinite(values)]
    fig, ax = plt.subplots()
    ax.hist(values, bins=bins); ax.set_xlabel("patch risk / confidence"); ax.set_ylabel("shots")
    return fig, ax

def plot_adaptive_diagnostics(table):
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    axes[0, 0].plot(table.short_rounds, table.pair_fallback_rate, "o"); axes[0, 0].set_ylabel("pair fallback rate")
    axes[0, 1].plot(table.short_rounds, table.mean_effective_rounds, "o"); axes[0, 1].set_ylabel("effective rounds")
    axes[1, 0].plot(table.pair_fallback_rate, table.logical_error_rate, "o"); axes[1, 0].set_xlabel("pair fallback rate"); axes[1, 0].set_ylabel("LER")
    for column in ["z_only_fallback_fraction", "x_only_fallback_fraction", "both_fallback_fraction"]:
        axes[1, 1].plot(table.short_rounds, table[column], "o", label=column)
    axes[1, 1].legend()
    return fig, axes

## Validation cells

Run this cell before either sweep. It deliberately uses tiny configurations. It raises immediately if a required endpoint, synchronization, prefix, or fallback-cause invariant fails.

In [ ]:
def validate_small_configuration():
    p = 0.0; distance = 3; rounds = 2
    parity_checks = get_parity_check_matrices("surface", distance)

    # F1: fixed MWPM and fixed BP-LSD at p=0.
    mwpm = run_fixed_point(p, distance, rounds, "mwpm", max_shots=2)
    bplsd = run_fixed_point(p, distance, rounds, "bplsd", max_shots=2)
    assert mwpm["logical_errors"] == 0
    assert bplsd["logical_errors"] == 0

    # F2: forced endpoint errors agree with the corresponding fixed paths.
    online, offline, matchable = decoder_generators("mwpm", p)
    fixed_short = knill_online_offline(parity_checks, 1, online, offline, matchable, p, 2, 10, PAULI, 1, surface_code=True)
    fixed_long = knill_online_offline(parity_checks, distance, online, offline, matchable, p, 2, 10, PAULI, 1, surface_code=True)
    for policy, expected in [(AlwaysShortPolicy(), fixed_short), (AlwaysLongPolicy(), fixed_long)]:
        endpoint = knill_online_offline_adaptive(
            parity_checks, AdaptiveSERounds(1, distance, policy), online, offline, matchable,
            p, 2, 10, PAULI, 1, batch_size=2, seed=stable_seed("endpoint", type(policy).__name__),
            surface_code=True, detail_level="analysis",
        )
        assert endpoint.logical_errors == expected[1] == 0

    # F3/F4: real threshold switching, long_rounds == distance, and event count.
    switched = run_adaptive_point(0.1, distance, 1, 0.01, "max_all_components", max_shots=4)
    assert switched["long_rounds"] == distance
    assert switched["pair_fallback_rate"] > 0

    # F5: the four patch-level truth-table cases reduce to an OR pair decision.
    truth = {(False, False): False, (True, False): True, (False, True): True, (True, True): True}
    assert all((z or x) == pair for (z, x), pair in truth.items())

    # F6: a selected individual long state-prep history retains its short prefix.
    schedule = AdaptiveSERounds(1, distance, ClusterLLRPolicy(0.01))
    description = generate_adaptive_state_prep_module(
        parity_checks, schedule, "z", 0.1, list(range(17)),
        make_bplsd_decoder_generator(0.1, alpha=CLUSTER_LLR_ALPHA, **BPLSD_OPTIONS),
        False, surface_code=True, confidence_aggregator=max_all_components,
    )
    execution = StatefulAdaptiveStatePrepExecutor().execute(description, batch_size=4, seed=stable_seed("prefix"))
    for shot, selected in enumerate(execution.selected_measurements):
        if execution.used_long[shot]:
            np.testing.assert_array_equal(selected[:description.short_circuit.num_measurements], execution.short_measurements[shot])

    # F8: fallback causes partition pair-level long events.
    parity_checks = get_parity_check_matrices("surface", 3)
    adaptive_result = knill_online_offline_adaptive(
        parity_checks, AdaptiveSERounds(1, 2, ClusterLLRPolicy(0.01)),
        pymatching.Matching.from_check_matrix,
        make_bplsd_decoder_generator(0.1, alpha=CLUSTER_LLR_ALPHA, **BPLSD_OPTIONS),
        False, 0.1, 4, 10, PAULI, 1, confidence_aggregator=max_all_components,
        batch_size=4, seed=stable_seed("cause"), detail_level="analysis", surface_code=True,
    )
    for pair in adaptive_result.bell_pair_stats:
        assert pair.long_count == pair.z_only_count + pair.x_only_count + pair.both_count
    assert len(adaptive_result.state_prep_stats) == 2 * NUM_TELEPORTATIONS
    assert len(adaptive_result.bell_pair_stats) == NUM_TELEPORTATIONS
    if adaptive_result.per_shot is not None:
        used = adaptive_result.per_shot["used_long"]
        assert np.all(used[:, 0::2] == used[:, 1::2])
        assert adaptive_result.per_shot["used_long_pair"].shape[1] == NUM_TELEPORTATIONS

    print("All small fixed/adaptive validation checks passed.")

# Execute only this tiny validation before choosing to run a sweep.
validate_small_configuration()

In [ ]:
# Benchmark only; this cell does not start a production sweep.
def estimate_smoke_runtime():
    start = time.perf_counter()
    _ = run_adaptive_point(0.01, 3, 1, 0.01, "max_all_components", max_shots=2)
    elapsed = time.perf_counter() - start
    points = len(PHYSICAL_ERROR_RATES) * len(DISTANCES) * len(SHORT_ROUNDS) * len(THRESHOLDS) * len(CONFIDENCE_AGGREGATORS)
    print({"two_shot_seconds": elapsed, "rough_points": points, "rough_seconds": elapsed * points / 2})

# Run only when you explicitly want a timing estimate.
# estimate_smoke_runtime()

In [ ]:
# Explicit opt-in sweep calls. They are not executed automatically.
# fixed_table = run_fixed_sweep(overwrite=False)
# adaptive_table = run_adaptive_sweep(overwrite=False)
# fig, ax = plot_adaptive_effective_rounds(adaptive_table)
# fig.savefig(FIGURES_ROOT / "adaptive_ler_vs_effective_rounds.png", dpi=150, bbox_inches="tight")